# Level 2: Generalization Test (Real Training with Validation Split)

**Goal**: Test if networks generalize by holding out validation samples during real training.

**Approach**:
1. Run actual DeepCFR training
2. Siphon off ~20% of samples BEFORE training (validation set)
3. Train networks on remaining 80% (training set)
4. Compare performance on training vs validation samples

**Pass Criteria**:
- Validation loss decreases during training (generalization)
- Validation loss doesn't diverge wildly from training loss
- Both losses decrease as iterations progress

**If it fails**: Overfitting, need regularization, or architecture issues.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import torch.optim as optim
import numpy as np
import random
from tqdm import tqdm
import copy

from network.model import DeepCFRModule
from core.trainer import DeepCFRTrainer, TrainingSample
from core.mccfr import MCCFR
from core.deep_cfr import DeepCFR

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version: 2.9.1
Device: cpu


## Step 1: Configuration

In [2]:
# Test configuration - FAST for quick iteration
# Tune these to balance speed vs quality
CONFIG = {
    'iterations': 3,            # Number of CFR iterations (start small!)
    'traversals_per_iter': 50,  # Traversals per iteration (paper: 10,000) - REDUCE for speed
    'sgd_iterations': 200,      # SGD steps per training (paper: 4,000) - REDUCE for speed
    'network_dim': 64,          # Network hidden dimension - smaller = faster
    'batch_size': 128,          # Training batch size
    'learning_rate': 0.001,     # Learning rate
    'val_fraction': 0.2,        # Fraction of samples to hold out for validation
}

# Estimate time
est_time_per_iter = CONFIG['traversals_per_iter'] * 0.1 + CONFIG['sgd_iterations'] * 0.01  # rough seconds
print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nEstimated time per iteration: ~{est_time_per_iter:.0f}s")
print(f"Total estimated time: ~{est_time_per_iter * CONFIG['iterations'] / 60:.1f} min")

Configuration:
  iterations: 3
  traversals_per_iter: 50
  sgd_iterations: 200
  network_dim: 64
  batch_size: 128
  learning_rate: 0.001
  val_fraction: 0.2

Estimated time per iteration: ~7s
Total estimated time: ~0.3 min


## Step 2: Create Modified DeepCFR with Validation Split

We subclass DeepCFR to intercept samples and hold out a validation set.

In [3]:
class DeepCFRWithValidation(DeepCFR):
    """
    DeepCFR that holds out validation samples during training.
    
    Intercepts samples before they go to training, randomly assigns
    some to validation set which is never trained on.
    """
    
    def __init__(self, val_fraction=0.2, **kwargs):
        super().__init__(**kwargs)
        self.val_fraction = val_fraction
        
        # Validation sample storage (per player + strategy)
        self.val_samples = {
            0: [],  # Player 0 value network validation
            1: [],  # Player 1 value network validation
            'strategy': []  # Strategy network validation
        }
        
        # Track validation losses over iterations
        self.val_losses = {
            0: [],
            1: [],
            'strategy': []
        }
        
    def _add_sample_with_validation(self, trainer, sample, player_or_strategy):
        """
        Add sample to either training or validation set.
        
        Randomly assigns samples to validation with probability val_fraction.
        """
        if random.random() < self.val_fraction:
            # Hold out for validation
            self.val_samples[player_or_strategy].append(sample)
        else:
            # Add to training
            trainer.add_samples([sample])
    
    def compute_validation_loss(self, network, trainer, val_samples):
        """
        Compute loss on validation samples without training.
        """
        if len(val_samples) == 0:
            return float('nan')
            
        network.eval()
        
        # Prepare validation batch
        cc_list, ah_list, target_batch, iter_weights, legal_mask = trainer.prepare_batch(val_samples)
        
        with torch.no_grad():
            predictions = network(cc_list, ah_list)
            squared_errors = (predictions - target_batch) ** 2
            masked_errors = squared_errors * legal_mask
            per_sample_loss = masked_errors.sum(dim=1)
            
            # Apply LCFR weighting
            T = max(self.iteration_count, 1)
            scaled_weights = iter_weights * (2.0 / T)
            loss = (scaled_weights * per_sample_loss).mean()
        
        network.train()
        return loss.item()
    
    def run_iteration_with_validation(self, use_network=None, progress_callback=None):
        """
        Run iteration but track validation loss before and after training.
        """
        # Record sample counts before iteration
        train_counts_before = {
            0: len(self.trainers[0].samples),
            1: len(self.trainers[1].samples),
            'strategy': len(self.strategy_trainer.samples)
        }
        
        # Run the standard iteration (this collects samples and trains)
        result = self.run_iteration(use_network=use_network, progress_callback=progress_callback)
        
        # Compute validation losses after training
        val_loss_v0 = self.compute_validation_loss(
            self.networks[0], self.trainers[0], self.val_samples[0]
        )
        val_loss_v1 = self.compute_validation_loss(
            self.networks[1], self.trainers[1], self.val_samples[1]
        )
        val_loss_strategy = self.compute_validation_loss(
            self.strategy_network, self.strategy_trainer, self.val_samples['strategy']
        )
        
        self.val_losses[0].append(val_loss_v0)
        self.val_losses[1].append(val_loss_v1)
        self.val_losses['strategy'].append(val_loss_strategy)
        
        result['val_loss_v0'] = val_loss_v0
        result['val_loss_v1'] = val_loss_v1
        result['val_loss_strategy'] = val_loss_strategy
        result['val_samples'] = {
            0: len(self.val_samples[0]),
            1: len(self.val_samples[1]),
            'strategy': len(self.val_samples['strategy'])
        }
        
        return result

print("DeepCFRWithValidation class defined.")

DeepCFRWithValidation class defined.


## Step 3: Monkey-patch Sample Collection to Support Validation Split

We need to intercept where samples are added to trainers.

In [4]:
# We'll patch the MCCFR class to support validation splitting
# The key is to intercept add_samples() calls

class ValidationTrainer(DeepCFRTrainer):
    """
    Trainer that randomly holds out samples for validation.
    """
    
    def __init__(self, val_fraction=0.2, **kwargs):
        super().__init__(**kwargs)
        self.val_fraction = val_fraction
        self.val_samples = []  # Validation samples (never trained on)
        
    def add_sample(self, sample):
        """
        Override add_sample (singular) to randomly split into train/val.
        deep_cfr.py calls add_sample(), not add_samples().
        """
        if random.random() < self.val_fraction:
            self.val_samples.append(sample)
        else:
            super().add_sample(sample)
    
    def add_samples(self, new_samples):
        """Override add_samples to use our add_sample for each."""
        for sample in new_samples:
            self.add_sample(sample)
    
    def compute_val_loss(self, network):
        """
        Compute loss on validation samples.
        """
        if len(self.val_samples) == 0:
            return float('nan')
            
        network.eval()
        
        cc_list, ah_list, target_batch, iter_weights, legal_mask = self.prepare_batch(self.val_samples)
        
        with torch.no_grad():
            predictions = network(cc_list, ah_list)
            squared_errors = (predictions - target_batch) ** 2
            masked_errors = squared_errors * legal_mask
            per_sample_loss = masked_errors.sum(dim=1)
            loss = per_sample_loss.mean()  # Simple mean for validation
        
        network.train()
        return loss.item()

print("ValidationTrainer class defined.")

ValidationTrainer class defined.


## Step 4: Create DeepCFR with Validation Trainers

In [5]:
# Create DeepCFR instance
deep_cfr = DeepCFR(
    network_dim=CONFIG['network_dim'],
    learning_rate=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    traversals_per_iter=CONFIG['traversals_per_iter'],
    sgd_iterations=CONFIG['sgd_iterations'],
    use_network_after=1,  # Use network from iteration 1
)

# Replace trainers with ValidationTrainers
val_fraction = CONFIG['val_fraction']

deep_cfr.trainers[0] = ValidationTrainer(
    val_fraction=val_fraction,
    network=deep_cfr.networks[0],
    mccfr=deep_cfr.mccfr,
    learning_rate=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    sgd_iterations=CONFIG['sgd_iterations']
)

deep_cfr.trainers[1] = ValidationTrainer(
    val_fraction=val_fraction,
    network=deep_cfr.networks[1],
    mccfr=deep_cfr.mccfr,
    learning_rate=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    sgd_iterations=CONFIG['sgd_iterations']
)

deep_cfr.strategy_trainer = ValidationTrainer(
    val_fraction=val_fraction,
    network=deep_cfr.strategy_network,
    mccfr=deep_cfr.mccfr,
    learning_rate=CONFIG['learning_rate'],
    batch_size=CONFIG['batch_size'],
    sgd_iterations=CONFIG['sgd_iterations']
)

# Update references
deep_cfr.trainer = deep_cfr.trainers[0]

print(f"DeepCFR created with {val_fraction*100:.0f}% validation split")
print(f"Parameters per network: {sum(p.numel() for p in deep_cfr.networks[0].parameters()):,}")

DeepCFR created with 20% validation split
Parameters per network: 123,787


## Step 5: Run Training with Validation Tracking

In [6]:
import time
import matplotlib.pyplot as plt

# Track metrics
history = {
    'iteration': [],
    'train_loss_v0': [],
    'train_loss_v1': [],
    'train_loss_strategy': [],
    'val_loss_v0': [],
    'val_loss_v1': [],
    'val_loss_strategy': [],
    'train_samples_v0': [],
    'train_samples_v1': [],
    'train_samples_strategy': [],
    'val_samples_v0': [],
    'val_samples_v1': [],
    'val_samples_strategy': [],
    'time_per_iter': [],
}

def plot_progress():
    """Live plot of training progress after each iteration."""
    if len(history['iteration']) == 0:
        return
        
    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    iters = history['iteration']
    
    for ax, name, train_key, val_key in [
        (axes[0], "V0", 'train_loss_v0', 'val_loss_v0'),
        (axes[1], "V1", 'train_loss_v1', 'val_loss_v1'),
        (axes[2], "Π", 'train_loss_strategy', 'val_loss_strategy'),
    ]:
        train_losses = [x for x in history[train_key] if not np.isnan(x)]
        val_losses = [x for x in history[val_key] if not np.isnan(x)]
        
        if train_losses:
            ax.plot(iters[:len(train_losses)], train_losses, 'b-o', label='Train', markersize=6)
        if val_losses:
            ax.plot(iters[:len(val_losses)], val_losses, 'r-s', label='Val', markersize=6)
        
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Loss')
        ax.set_title(f'{name}')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        if train_losses or val_losses:
            ax.set_yscale('log')
    
    plt.tight_layout()
    plt.show()

def progress_callback(completed, total, phase):
    """Callback for progress during iteration."""
    pct = completed / total * 100
    bar_len = 30
    filled = int(bar_len * completed / total)
    bar = '█' * filled + '░' * (bar_len - filled)
    print(f"\r  [{bar}] {pct:5.1f}% - {phase}: {completed}/{total}", end='', flush=True)

print(f"Running {CONFIG['iterations']} iterations...")
print(f"Config: {CONFIG['traversals_per_iter']} traversals, {CONFIG['sgd_iterations']} SGD steps")
print("="*70)

total_start = time.time()

for i in range(CONFIG['iterations']):
    iter_start = time.time()
    print(f"\n▶ Iteration {i+1}/{CONFIG['iterations']}")
    
    # Run iteration with progress callback
    result = deep_cfr.run_iteration(progress_callback=progress_callback)
    print()  # newline after progress bar
    
    iter_time = time.time() - iter_start
    history['time_per_iter'].append(iter_time)
    
    # Record training metrics
    history['iteration'].append(i + 1)
    history['train_loss_v0'].append(result.get('loss_p0', float('nan')))
    history['train_loss_v1'].append(result.get('loss_p1', float('nan')))
    history['train_loss_strategy'].append(result.get('loss_strategy', float('nan')))
    
    # Compute validation losses
    val_loss_v0 = deep_cfr.trainers[0].compute_val_loss(deep_cfr.networks[0])
    val_loss_v1 = deep_cfr.trainers[1].compute_val_loss(deep_cfr.networks[1])
    val_loss_strategy = deep_cfr.strategy_trainer.compute_val_loss(deep_cfr.strategy_network)
    
    history['val_loss_v0'].append(val_loss_v0)
    history['val_loss_v1'].append(val_loss_v1)
    history['val_loss_strategy'].append(val_loss_strategy)
    
    # Sample counts
    history['train_samples_v0'].append(len(deep_cfr.trainers[0].samples))
    history['train_samples_v1'].append(len(deep_cfr.trainers[1].samples))
    history['train_samples_strategy'].append(len(deep_cfr.strategy_trainer.samples))
    history['val_samples_v0'].append(len(deep_cfr.trainers[0].val_samples))
    history['val_samples_v1'].append(len(deep_cfr.trainers[1].val_samples))
    history['val_samples_strategy'].append(len(deep_cfr.strategy_trainer.val_samples))
    
    # Print summary table
    print(f"  ┌{'─'*66}┐")
    print(f"  │ {'Network':<8} │ {'Train Loss':<12} │ {'Val Loss':<12} │ {'Train':<8} │ {'Val':<8} │")
    print(f"  ├{'─'*66}┤")
    print(f"  │ {'V0':<8} │ {result.get('loss_p0', 0):<12.4f} │ {val_loss_v0:<12.4f} │ {len(deep_cfr.trainers[0].samples):<8} │ {len(deep_cfr.trainers[0].val_samples):<8} │")
    print(f"  │ {'V1':<8} │ {result.get('loss_p1', 0):<12.4f} │ {val_loss_v1:<12.4f} │ {len(deep_cfr.trainers[1].samples):<8} │ {len(deep_cfr.trainers[1].val_samples):<8} │")
    print(f"  │ {'Π':<8} │ {result.get('loss_strategy', 0):<12.4f} │ {val_loss_strategy:<12.4f} │ {len(deep_cfr.strategy_trainer.samples):<8} │ {len(deep_cfr.strategy_trainer.val_samples):<8} │")
    print(f"  └{'─'*66}┘")
    print(f"  ⏱ Iteration time: {iter_time:.1f}s")
    
    # Show live plot after each iteration
    plot_progress()

total_time = time.time() - total_start
print("\n" + "="*70)
print(f"✓ Training complete!")
print(f"  Total time: {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"  Avg per iteration: {np.mean(history['time_per_iter']):.1f}s")

Running 3 iterations...
Config: 50 traversals, 200 SGD steps

▶ Iteration 1/3
  [███████████████░░░░░░░░░░░░░░░]  50.0% - samples: 50/100

KeyboardInterrupt: 

## Step 6: Visualize Training vs Validation Loss

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: Loss curves
for ax, name, train_key, val_key in [
    (axes[0, 0], "V0 (Player 0)", 'train_loss_v0', 'val_loss_v0'),
    (axes[0, 1], "V1 (Player 1)", 'train_loss_v1', 'val_loss_v1'),
    (axes[0, 2], "Π (Strategy)", 'train_loss_strategy', 'val_loss_strategy'),
]:
    train_losses = [x for x in history[train_key] if not np.isnan(x)]
    val_losses = [x for x in history[val_key] if not np.isnan(x)]
    
    if train_losses:
        ax.plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='Training', markersize=4)
    if val_losses:
        ax.plot(range(1, len(val_losses)+1), val_losses, 'r-s', label='Validation', markersize=4)
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title(f'{name} Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    if train_losses or val_losses:
        ax.set_yscale('log')

# Bottom row: Sample counts
for ax, name, train_key, val_key in [
    (axes[1, 0], "V0", 'train_samples_v0', 'val_samples_v0'),
    (axes[1, 1], "V1", 'train_samples_v1', 'val_samples_v1'),
    (axes[1, 2], "Π", 'train_samples_strategy', 'val_samples_strategy'),
]:
    ax.plot(history['iteration'], history[train_key], 'b-o', label='Training', markersize=4)
    ax.plot(history['iteration'], history[val_key], 'r-s', label='Validation', markersize=4)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Sample Count')
    ax.set_title(f'{name} Samples')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Final Analysis

In [ ]:
print("="*70)
print("GENERALIZATION TEST ANALYSIS")
print("="*70)

def analyze_network(name, train_losses, val_losses, train_samples, val_samples):
    """Analyze a single network's generalization."""
    print(f"\n{name}:")
    
    # Filter NaN values
    train_losses = [x for x in train_losses if not np.isnan(x)]
    val_losses = [x for x in val_losses if not np.isnan(x)]
    
    if len(train_losses) < 2 or len(val_losses) < 2:
        print("  Not enough data points for analysis")
        return None
    
    # Training loss trend
    train_decreased = train_losses[-1] < train_losses[0]
    train_reduction = (1 - train_losses[-1]/train_losses[0])*100 if train_losses[0] > 0 else 0
    
    # Validation loss trend
    val_decreased = val_losses[-1] < val_losses[0]
    val_reduction = (1 - val_losses[-1]/val_losses[0])*100 if val_losses[0] > 0 else 0
    
    # Overfitting check: is val loss much higher than train loss?
    final_ratio = val_losses[-1] / (train_losses[-1] + 1e-8)
    overfitting = final_ratio > 5  # Val more than 5x worse than train
    
    print(f"  Training:   {train_losses[0]:.4f} → {train_losses[-1]:.4f} ({train_reduction:+.1f}%)")
    print(f"  Validation: {val_losses[0]:.4f} → {val_losses[-1]:.4f} ({val_reduction:+.1f}%)")
    print(f"  Final val/train ratio: {final_ratio:.2f}x")
    print(f"  Samples: {train_samples[-1]} train, {val_samples[-1]} val")
    
    # Verdict
    if val_decreased and not overfitting:
        print(f"  Status: PASS (generalizing)")
        return True
    elif val_decreased and overfitting:
        print(f"  Status: PARTIAL (learning but overfitting)")
        return False
    else:
        print(f"  Status: FAIL (not generalizing)")
        return False

results = [
    analyze_network("V0 (Player 0)", history['train_loss_v0'], history['val_loss_v0'],
                   history['train_samples_v0'], history['val_samples_v0']),
    analyze_network("V1 (Player 1)", history['train_loss_v1'], history['val_loss_v1'],
                   history['train_samples_v1'], history['val_samples_v1']),
    analyze_network("Π (Strategy)", history['train_loss_strategy'], history['val_loss_strategy'],
                   history['train_samples_strategy'], history['val_samples_strategy']),
]

# Filter None results
results = [r for r in results if r is not None]

print("\n" + "="*70)
if len(results) == 0:
    print("OVERALL: INCONCLUSIVE - Not enough data")
elif all(results):
    print("OVERALL: PASS - All networks generalize!")
    print("Validation loss decreased along with training loss.")
    print("Networks learn patterns, not just memorize training data.")
elif any(results):
    print("OVERALL: PARTIAL - Some networks generalize, some don't.")
    print("Check failing networks for overfitting issues.")
else:
    print("OVERALL: FAIL - Networks are not generalizing.")
    print("\nPossible issues:")
    print("  1. Overfitting to training samples")
    print("  2. Need more iterations")
    print("  3. Need regularization (dropout, weight decay)")
    print("  4. Learning rate too high")
print("="*70)

## Step 8: Detailed Validation Sample Analysis

In [ ]:
def analyze_prediction_quality(network, trainer, samples, name):
    """Analyze how well predictions match targets."""
    if len(samples) == 0:
        print(f"{name}: No samples")
        return
        
    network.eval()
    cc_list, ah_list, target_batch, _, legal_mask = trainer.prepare_batch(samples)
    
    with torch.no_grad():
        predictions = network(cc_list, ah_list)
    
    errors = (predictions - target_batch).abs() * legal_mask
    max_errors = errors.max(dim=1)[0]
    mean_errors = errors.sum(dim=1) / (legal_mask.sum(dim=1) + 1e-8)
    
    print(f"\n{name} ({len(samples)} samples):")
    print(f"  Max error:  {max_errors.max().item():.4f}")
    print(f"  Mean error: {mean_errors.mean().item():.4f}")
    print(f"  Samples < 0.5 error: {(max_errors < 0.5).sum().item()}/{len(samples)} ({(max_errors < 0.5).float().mean()*100:.1f}%)")
    print(f"  Samples < 1.0 error: {(max_errors < 1.0).sum().item()}/{len(samples)} ({(max_errors < 1.0).float().mean()*100:.1f}%)")

print("="*70)
print("PREDICTION QUALITY ON TRAINING SAMPLES")
print("="*70)
analyze_prediction_quality(deep_cfr.networks[0], deep_cfr.trainers[0], 
                          deep_cfr.trainers[0].samples, "V0 Training")
analyze_prediction_quality(deep_cfr.networks[1], deep_cfr.trainers[1], 
                          deep_cfr.trainers[1].samples, "V1 Training")
analyze_prediction_quality(deep_cfr.strategy_network, deep_cfr.strategy_trainer, 
                          deep_cfr.strategy_trainer.samples, "Π Training")

print("\n" + "="*70)
print("PREDICTION QUALITY ON VALIDATION SAMPLES (GENERALIZATION)")
print("="*70)
analyze_prediction_quality(deep_cfr.networks[0], deep_cfr.trainers[0], 
                          deep_cfr.trainers[0].val_samples, "V0 Validation")
analyze_prediction_quality(deep_cfr.networks[1], deep_cfr.trainers[1], 
                          deep_cfr.trainers[1].val_samples, "V1 Validation")
analyze_prediction_quality(deep_cfr.strategy_network, deep_cfr.strategy_trainer, 
                          deep_cfr.strategy_trainer.val_samples, "Π Validation")